In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import glob
import os
import numpy as np

In [8]:
species_list = ['ath', 'osa', 'zma', 'sly', 'gar']

for species in species_list:
    # ====== 输入文件 ======
    info_file = "%s/rRNA.info" %species
    depth_file = "%s.csv" %species
    output_file = "%s.rRNA_ratio.csv" %species
   
    # 读取区间
    info_df = pd.read_csv(info_file, sep="\t")
    target_types = ["rRNA_5S", "rRNA_18S", "rRNA_5_8S", "rRNA_25S"]
    info_df = info_df[info_df["type"].isin(target_types)]

    depth_df = pd.read_csv(depth_file, index_col=0)

    # 转 numpy
    depth_values = depth_df.values

    # 计算总深度
    total_depth = 0
    for _, row in info_df.iterrows():
        start = int(row["start_pos"])
        end = int(row["end_pos"])
        total_depth += depth_values[:, start:end+1].sum(axis=1)

    # 计算比例
    result = pd.DataFrame(index=depth_df.index)

    for _, row in info_df.iterrows():
        region = row["type"]
        start = int(row["start_pos"])
        end = int(row["end_pos"])
        region_depth = depth_values[:, start:end+1].sum(axis=1)
        result[region + "_ratio"] = region_depth / total_depth

    result.to_csv(output_file)

    print("Done.")

Done.
Done.
Done.
Done.
Done.


In [21]:
plt.rcParams["font.family"] = "DejaVu Sans"
files = glob.glob("*rRNA_ratio.csv")

if len(files) == 0:
    print("No rRNA_ratio.csv files found.")
    exit()

output_pdf = "rRNA_stacked_bar_by_species.pdf"

# 设置颜色
rRNA_colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]  

with PdfPages(output_pdf) as pdf:
    for file in files:
        species = os.path.basename(file).split(".")[0]
        df = pd.read_csv(file, index_col=0)
        
        plt.figure(figsize=(10, 6))
        
        bottom = np.zeros(len(df))
        for i, col in enumerate(df.columns):
            plt.bar(df.index, df[col], bottom=bottom, color=rRNA_colors[i], label=col)
            bottom += df[col].values
        
        plt.xticks(rotation=90)
        plt.ylabel("Proportion")
        plt.ylim(0, 1)
        plt.title(f"{species} rRNA composition")
        plt.legend(title="rRNA type", bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        
        pdf.savefig(dpi=300)
        plt.close()

print("PDF generated:", output_pdf)

PDF generated: rRNA_stacked_bar_by_species.pdf
